# <center> <font size = 24 color = 'steelblue'>**Nova Haven - UrbanPulse Analytics**
**Complaint Classification and Routing Recommendations**

### Model 4: 311 Complaint Classification — NLP

- Multi-class text classification: predict the complaint type from text
- **Top 5 categories + "Other" (6 classes total):** Illegal Parking, HEAT/HOT WATER, Noise - Residential, Snow or Ice, Blocked Driveway, and Other (everything else)
- Use NLP techniques (TF-IDF + classifier, embeddings, or transformers)
- Handle real-world citizen text (informal language, abbreviations, multiple languages)
- **Minimum Benchmark:** Accuracy > 75%, weighted F1 > 0.70
- **Stretch Goal:** Accuracy > 85%, weighted F1 > 0.80
- **Also valued:** Complaint routing recommendations

In [99]:
"""
Smart City Data Preprocessing Hints
=====================================
These are HINTS, not complete solutions. Use them as a starting point
for your data pipeline. You'll need to adapt and expand these for your
specific models.

Datasets:
- city_traffic_accidents.csv (~500K accident records)
- pothole_images/ (~4,400 road surface images)
- urbanpulse_311_complaints.csv (~500K complaint records)
"""

# =============================================================================
# *** CLASS IMBALANCE WARNING ***
# =============================================================================
# EVERY dataset in this project has class imbalance. If you ignore it,
# your model will learn to predict the majority class and look "accurate"
# while being clinically useless.
#
# Techniques you MUST consider for every model:
#   1. class_weight='balanced' in sklearn models (easiest first step)
#   2. SMOTE (Synthetic Minority Oversampling) from imblearn
#   3. Stratified train/test splits (use stratify= in train_test_split)
#   4. Weighted loss functions in TensorFlow/Keras
#   5. Evaluation with weighted F1, precision, recall — NOT just accuracy
#
# A model that predicts the majority class for everything is WORTHLESS
# even if it gets 80%+ accuracy. Always check per-class metrics.
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

In [100]:
import sys

project_root = Path.cwd().parent  # or adjust levels
sys.path.insert(0, str(project_root))

In [101]:
# Project paths
PROJECT_ROOT = project_root #Path(__file__).resolve().parents[1]
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

In [102]:
#filepath = RAW_DATA_DIR / "urbanpulse_11_complaints.csv"
#df = pd.read_csv(filepath)

In [103]:
#!pip install --upgrade datasets huggingface_hub

In [104]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [105]:
colab_filepath = '/content/drive/MyDrive/cleaned_data/urbanpulse_311_complaints.csv'

In [106]:
df = pd.read_csv(colab_filepath)

In [107]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 434722 entries, 0 to 434721
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   unique_key              434722 non-null  int64 
 1   created_date            434722 non-null  object
 2   closed_date             384642 non-null  object
 3   agency                  434722 non-null  object
 4   agency_name             434722 non-null  object
 5   complaint_type          434722 non-null  object
 6   descriptor              431047 non-null  object
 7   resolution_description  434722 non-null  object
 8   borough                 434722 non-null  object
 9   open_data_channel_type  434722 non-null  object
 10  status                  434722 non-null  object
dtypes: int64(1), object(10)
memory usage: 36.5+ MB


In [108]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Sklearn - evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

# Model saving
import joblib

# Settings
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("Libraries imported successfully!")


Libraries imported successfully!


In [109]:
# -----------------------------
# 1. Reproducibility
# -----------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [110]:
df = df.apply(lambda col: col.str.lower() if col.dtype == 'object' else col)

In [111]:


# -----------------------------
# 2. Basic text cleaning
# -----------------------------
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)  # collapse whitespace
    return text

In [112]:
def get_top_complaint_types(df: pd.DataFrame, n: int = 5) -> list:
    """
    Focus your NLP model on the top 5 complaint types + an "Other" bucket.

    With 186 categories, many have very few examples. Focusing on the top 5
    keeps your model tractable while covering the highest-impact complaint
    types. The "Other" bucket captures the long tail.

    The top 5 311 categories typically cover about 50-60% of complaints.

    The top 5 categories in this dataset are:
    - Illegal Parking
    - HEAT/HOT WATER
    - Noise - Residential
    - Snow or Ice
    - Blocked Driveway
    """
    top_types = df['complaint_type'].value_counts().head(n).index.tolist()
    coverage = df[df['complaint_type'].isin(top_types)].shape[0] / len(df) * 100
    print(f"Top {n} complaint types cover {coverage:.1f}% of all complaints")
    return top_types

In [113]:
get_top_complaint_types(df)

Top 5 complaint types cover 51.0% of all complaints


['illegal parking',
 'heat/hot water',
 'noise - residential',
 'snow or ice',
 'blocked driveway']

In [114]:
def create_complaint_categories(df: pd.DataFrame) -> pd.DataFrame:
    """
    Map complaint types to the top 5 categories + "Other" (6 classes total).

    The top 5 categories are:
    - Illegal Parking
    - HEAT/HOT WATER
    - Noise - Residential
    - Snow or Ice
    - Blocked Driveway

    Everything else maps to "Other". This gives you 6 classes total —
    a much more manageable classification problem than 186 categories.
    """
    top_5 = ['illegal parking', 'heat/hot water', 'noise - residential',
             'snow or ice', 'blocked driveway']
    df['complaint_category'] = df['complaint_type'].apply(
        lambda x: x if x in top_5 else 'other'
    )

    print("Complaint category distribution:")
    print(df['complaint_category'].value_counts())

    coverage = df[df['complaint_category'] != 'other'].shape[0] / len(df) * 100
    print(f"\nTop 5 categories cover {coverage:.1f}% of all complaints")
    print(f"Total classes: {df['complaint_category'].nunique()} (top 5 + other)")

    return df

In [115]:
df_clean = create_complaint_categories(df)

Complaint category distribution:
complaint_category
other                  212897
illegal parking         66159
heat/hot water          64362
noise - residential     38931
snow or ice             27453
blocked driveway        24920
Name: count, dtype: int64

Top 5 categories cover 51.0% of all complaints
Total classes: 6 (top 5 + other)


In [116]:
df = df_clean


df["complaint_text"] = df["descriptor"].apply(clean_text)
df["categories"] = df['complaint_category']

# Drop empty text / labels
df = df[(df["complaint_text"] != "") & (df["categories"] != "")]
df = df.dropna(subset=["complaint_text", "categories"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["categories"].value_counts())

Dataset shape: (431047, 14)

Class distribution:
categories
other                  209222
illegal parking         66159
heat/hot water          64362
noise - residential     38931
snow or ice             27453
blocked driveway        24920
Name: count, dtype: int64


In [117]:
# -----------------------------
# 4. Encode labels
# -----------------------------
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["categories"])

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

print("\nLabel mapping:")
print(id2label)


Label mapping:
{0: 'blocked driveway', 1: 'heat/hot water', 2: 'illegal parking', 3: 'noise - residential', 4: 'other', 5: 'snow or ice'}


In [118]:
df[["complaint_text", "label"]].head(10)

,complaint_text,label
0,entire building,1
1,water supply,4
2,posted parking sign violation,2
3,entire building,1
4,blocked crosswalk,2
5,posted parking sign violation,2
6,loud music/party,4
7,apartment only,1
8,entire building,1
9,inadequate or no heat,4


TF-IDF + Logistic Regression

In [119]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [120]:



# -----------------------------
# 3. Split data
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df["complaint_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [121]:
from sklearn.pipeline import Pipeline

In [122]:


# -----------------------------
# 4. Build pipeline
# -----------------------------
clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight = 'balanced',
        n_jobs=-1
    ))
])

In [123]:

# -----------------------------
# 5. Train
# -----------------------------
clf.fit(X_train, y_train)

# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Accuracy: 0.965955225611878

Classification Report:
                     precision    recall  f1-score   support

   blocked driveway       1.00      1.00      1.00      4984
     heat/hot water       1.00      1.00      1.00     12872
    illegal parking       1.00      1.00      1.00     13232
noise - residential       0.73      1.00      0.84      7786
              other       1.00      0.93      0.96     41845
        snow or ice       1.00      1.00      1.00      5491

           accuracy                           0.97     86210
          macro avg       0.95      0.99      0.97     86210
       weighted avg       0.98      0.97      0.97     86210



In [124]:
### print performance metrics for this TF-IDF + Logistic Regression model:

y_train_pred = clf.predict(X_train)
y_test_pred = clf.predict(X_test)

print (f"'Train Accuracy': {accuracy_score(y_train, y_train_pred):.4f}")
print (f"'Test Accuracy': {accuracy_score(y_test, y_test_pred):.4f}")
print (f"'Precision (weighted)': {precision_score(y_test, y_test_pred, average='weighted'):.4f}")
print (f"'Recall (weighted)': {recall_score(y_test, y_test_pred, average='weighted'):.4f}")
print (f"'F1 (weighted)': {f1_score(y_test, y_test_pred, average='weighted'):.4f}")

'Train Accuracy': 0.9666
'Test Accuracy': 0.9660
'Precision (weighted)': 0.9753
'Recall (weighted)': 0.9660
'F1 (weighted)': 0.9680


# Build Complaint Routing Recommendations

In [125]:
print(df.groupby(['categories', 'agency'])['agency_name'].count())

categories           agency
blocked driveway     nypd      24920
heat/hot water       hpd       64362
illegal parking      nypd      66159
noise - residential  nypd      38931
other                dcwp       1915
                     dep       20719
                     dhs        1573
                     dob       11126
                     doe         328
                     dohmh      6388
                     dot       38642
                     dpr        7178
                     dsny      24029
                     hpd       63486
                     nypd      30586
                     oos           7
                     oti          17
                     tlc        3228
snow or ice          dsny      27453
Name: agency_name, dtype: int64


Five of the 6 categories already have a default route-to-agency:

**"blocked driveway"** --- nypd

**"heat/hot water"** --- hpd

**"illegal parking"** --- nypd

**"noise - residential"** --- nypd

**"snow or ice"** --- dsny

However, we need to be able to route the complaints in the **"other"** category to the correct responding agency.

Below we do a quick EDA on the complaints that were categorized" as **"other"**

In [126]:
df_other = df[df['label']==4]

In [127]:
df_other.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,resolution_description,borough,open_data_channel_type,status,complaint_category,complaint_text,categories,label
1,68207007,2026-03-04t10:40:04.000,2026-03-06t16:05:24.000,hpd,department of housing preservation and develop...,plumbing,water supply,hpd conducted an inspection of this complaint....,bronx,online,closed,other,water supply,other,4
6,67996182,2026-02-14t04:34:09.000,2026-02-14t07:30:31.000,nypd,new york city police department,noise - street/sidewalk,loud music/party,the new york city police department responded ...,brooklyn,mobile,closed,other,loud music/party,other,4
9,67918125,2026-02-09t11:41:22.000,NaN,dohmh,department of health and mental hygiene,non-residential heat,inadequate or no heat,the department of health and mental hygiene ha...,bronx,online,in progress,other,inadequate or no heat,other,4
10,68254352,2026-03-08t19:18:00.000,2026-03-09t12:22:00.000,dep,department of environmental protection,lead,lead kit request (residential) (l10),the department of environmental protection (de...,brooklyn,online,closed,other,lead kit request (residential) (l10),other,4
14,67928066,2026-02-10t11:11:28.000,2026-03-11t14:35:11.000,hpd,department of housing preservation and develop...,door/window,door,hpd conducted an inspection of this complaint....,brooklyn,phone,closed,other,door,other,4


In [128]:
# -----------------------------
# Encode labels for route-to-agency
# -----------------------------
label_encoder_rt = LabelEncoder()
df["rt_label"] = label_encoder_rt.fit_transform(df["agency"])

id2label_rt = {i: label for i, label in enumerate(label_encoder_rt.classes_)}
label2id_rt = {label: i for i, label in id2label.items()}

print("\nLabel mapping:")
print(id2label_rt)


Label mapping:
{0: 'dcwp', 1: 'dep', 2: 'dhs', 3: 'dob', 4: 'doe', 5: 'dohmh', 6: 'dot', 7: 'dpr', 8: 'dsny', 9: 'hpd', 10: 'nypd', 11: 'oos', 12: 'oti', 13: 'tlc'}


In [129]:
## use complaint_type to predict route_to_agency for "other" category

X_train_rt, X_test_rt, y_train_rt, y_test_rt = train_test_split(
    df["complaint_type"],
    df["rt_label"],
    test_size=0.2,
    random_state=42,
    stratify=df["rt_label"]
)

In [130]:


# -----------------------------
# 4. Build pipeline
# -----------------------------
clf_rt = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=50000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight = 'balanced',
        n_jobs=-1
    ))
])

In [131]:

# -----------------------------
# 5. Train
# -----------------------------
clf_rt.fit(X_train_rt, y_train_rt)

# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred_rt = clf_rt.predict(X_test_rt)

print("Accuracy:", accuracy_score(y_test_rt, y_pred_rt))
print("\nClassification Report:")
print(classification_report(y_test_rt, y_pred_rt, target_names=label_encoder_rt.classes_))

Accuracy: 0.9979468739125391

Classification Report:
              precision    recall  f1-score   support

        dcwp       1.00      1.00      1.00       383
         dep       1.00      1.00      1.00      4144
         dhs       1.00      1.00      1.00       315
         dob       0.98      0.96      0.97      2225
         doe       1.00      1.00      1.00        66
       dohmh       1.00      1.00      1.00      1278
         dot       1.00      1.00      1.00      7728
         dpr       1.00      1.00      1.00      1436
        dsny       1.00      1.00      1.00     10296
         hpd       1.00      1.00      1.00     25570
        nypd       1.00      1.00      1.00     32119
         oos       1.00      1.00      1.00         1
         oti       1.00      1.00      1.00         3
         tlc       1.00      1.00      1.00       646

    accuracy                           1.00     86210
   macro avg       1.00      1.00      1.00     86210
weighted avg       1.00    

In [132]:
y_train_pred_rt = clf_rt.predict(X_train_rt)
y_test_pred_rt = clf_rt.predict(X_test_rt)

In [138]:
### print performance metrics for this routing recommendation engine:

print (f"'Train Accuracy': {accuracy_score(y_train_rt, y_train_pred_rt):.4f}")
print (f"'Test Accuracy': {accuracy_score(y_test_rt, y_test_pred_rt):.4f}")
print (f"'Precision (weighted)': {precision_score(y_test_rt, y_test_pred_rt, average='weighted'):.4f}")
print (f"'Recall (weighted)': {recall_score(y_test_rt, y_test_pred_rt, average='weighted'):.4f}")
print (f"'F1 (weighted)': {f1_score(y_test_rt, y_test_pred_rt, average='weighted'):.4f}")

'Train Accuracy': 0.9978
'Test Accuracy': 0.9979
'Precision (weighted)': 0.9979
'Recall (weighted)': 0.9979
'F1 (weighted)': 0.9979


In [139]:
### map recommended labels to agency and agency name

df.loc[X_train_rt.index, 'routing_label'] = y_train_pred_rt
df.loc[X_test_rt.index, 'routing_label'] = y_test_pred_rt

In [140]:
df['route_to_agency'] = df['routing_label'].map(id2label_rt)


In [141]:
agency_map = df[['agency', 'agency_name']].drop_duplicates().set_index('agency')['agency_name']
df['route_to agency_name'] = df['route_to_agency'].map(agency_map)

In [142]:
df[['categories', 'routing_label', 'route_to_agency', 'route_to agency_name']].value_counts()

categories           routing_label  route_to_agency  route_to agency_name                              
illegal parking      10.0           nypd             new york city police department                       66159
heat/hot water       9.0            hpd              department of housing preservation and development    64362
other                9.0            hpd              department of housing preservation and development    63759
noise - residential  10.0           nypd             new york city police department                       38931
other                6.0            dot              department of transportation                          38657
                     10.0           nypd             new york city police department                       30435
snow or ice          8.0            dsny             department of sanitation                              27453
blocked driveway     10.0           nypd             new york city police department                       24920
other                8.0            dsny             department of sanitation                              24149
                     1.0            dep              department of environmental protection                20719
                     3.0            dob              department of buildings                               10855
                     7.0            dpr              department of parks and recreation                     7178
                     5.0            dohmh            department of health and mental hygiene                6402
                     13.0           tlc              taxi and limousine commission                          3228
                     0.0            dcwp             department of consumer and worker protection           1915
                     2.0            dhs              department of homeless services                        1573
                     4.0            doe              department of education                                 328
                     12.0           oti              office of technology and innovation                      17
                     11.0           oos              office of the sheriff                                     7
Name: count, dtype: int64

In [143]:
# Save NLP processed 311 Complaint data for visualization
#df.to_csv('../data/processed/predicted_311_complaint_data.csv', index=False)
df.to_csv('/content/drive/MyDrive/cleaned_data/predicted_311_complaint_data.csv')
print("✓ 311 Complaint data with routing recommendations saved to ../data/processed/predicted_311_complaint_data.csv")

✓ 311 Complaint data with routing recommendations saved to ../data/processed/predicted_311_complaint_data.csv
